In [1]:
import pandas as pd
import numpy as np
import re
import torch
import torch.nn as nn
import torch.optim as optim

from collections import Counter
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from torch.utils.data import Dataset, DataLoader

In [3]:
df = pd.read_csv("IMDB Dataset.csv")

print(df.shape)
df.head()

(50000, 2)


,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [4]:
df.isnull().sum()

review       0
sentiment    0
dtype: int64

In [5]:
df = df.drop_duplicates().reset_index(drop=True)

print(df.shape)

(49582, 2)


In [6]:
def clean_text(text):
    text = text.lower()

    # Remove HTML tags
    text = re.sub(r"<.*?>", " ", text)

    # Remove URLs
    text = re.sub(r"http\S+|www\S+", " ", text)

    # Keep letters and apostrophes
    text = re.sub(r"[^a-zA-Z']", " ", text)

    # Remove extra spaces
    text = re.sub(r"\s+", " ", text).strip()

    return text


df["review"] = df["review"].apply(clean_text)

df["sentiment"] = df["sentiment"].map({
    "negative": 0,
    "positive": 1
})

df.head()

,review,sentiment
0,one of the other reviewers has mentioned that ...,1
1,a wonderful little production the filming tech...,1
2,i thought this was a wonderful way to spend ti...,1
3,basically there's a family where a little boy ...,0
4,petter mattei's love in the time of money is a...,1


In [7]:
X_train, X_test, y_train, y_test = train_test_split(
    df["review"],
    df["sentiment"],
    test_size=0.2,
    random_state=42,
    stratify=df["sentiment"]
)

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))

Training samples: 39665
Testing samples: 9917


In [ ]:
# Build Vocabulary
def tokenize(text):
    return text.split()


counter = Counter()

for review in X_train:
    counter.update(tokenize(review))


MAX_VOCAB_SIZE = 20000

most_common_words = counter.most_common(MAX_VOCAB_SIZE - 2)

word_to_idx = {
    "<PAD>": 0,
    "<UNK>": 1
}

for word, _ in most_common_words:
    word_to_idx[word] = len(word_to_idx)


VOCAB_SIZE = len(word_to_idx)

print("Vocabulary size:", VOCAB_SIZE)

Vocabulary size: 20000


In [9]:
MAX_LENGTH = 200


def encode_review(text):
    tokens = tokenize(text)

    encoded = [
        word_to_idx.get(word, word_to_idx["<UNK>"])
        for word in tokens
    ]

    # Truncate
    encoded = encoded[:MAX_LENGTH]

    # Padding
    if len(encoded) < MAX_LENGTH:
        encoded += [word_to_idx["<PAD>"]] * (MAX_LENGTH - len(encoded))

    return encoded

In [10]:
class IMDBDataset(Dataset):

    def __init__(self, reviews, labels):
        self.reviews = reviews.reset_index(drop=True)
        self.labels = labels.reset_index(drop=True)

    def __len__(self):
        return len(self.reviews)

    def __getitem__(self, idx):

        encoded_review = encode_review(
            self.reviews.iloc[idx]
        )

        x = torch.tensor(
            encoded_review,
            dtype=torch.long
        )

        y = torch.tensor(
            self.labels.iloc[idx],
            dtype=torch.float32
        )

        return x, y

In [11]:
train_dataset = IMDBDataset(
    X_train,
    y_train
)

test_dataset = IMDBDataset(
    X_test,
    y_test
)

train_loader = DataLoader(
    train_dataset,
    batch_size=64,
    shuffle=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=64,
    shuffle=False
)

In [12]:
class SentimentRNN(nn.Module):

    def __init__(
        self,
        vocab_size,
        embedding_dim=128,
        hidden_size=128,
        num_layers=1
    ):

        super().__init__()

        self.embedding = nn.Embedding(
            vocab_size,
            embedding_dim,
            padding_idx=0
        )

        self.rnn = nn.RNN(
            input_size=embedding_dim,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True
        )

        self.fc = nn.Linear(
            hidden_size,
            1
        )


    def forward(self, x):

        # (batch, seq_len)
        x = self.embedding(x)

        # (batch, seq_len, embedding_dim)
        output, hidden = self.rnn(x)

        # Final hidden state
        final_hidden = hidden[-1]

        logits = self.fc(final_hidden)

        return logits.squeeze(1)

In [13]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", device)

Device: cpu


In [14]:
model = SentimentRNN(
    vocab_size=VOCAB_SIZE,
    embedding_dim=128,
    hidden_size=128
).to(device)

criterion = nn.BCEWithLogitsLoss()

optimizer = optim.Adam(
    model.parameters(),
    lr=0.001
)

model

SentimentRNN(
  (embedding): Embedding(20000, 128, padding_idx=0)
  (rnn): RNN(128, 128, batch_first=True)
  (fc): Linear(in_features=128, out_features=1, bias=True)
)

In [15]:
EPOCHS = 5

train_losses = []

for epoch in range(EPOCHS):

    model.train()

    total_loss = 0

    for xb, yb in train_loader:

        xb = xb.to(device)
        yb = yb.to(device)

        optimizer.zero_grad()

        logits = model(xb)

        loss = criterion(
            logits,
            yb
        )

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)

    train_losses.append(avg_loss)

    print(
        f"Epoch [{epoch+1}/{EPOCHS}] "
        f"Loss: {avg_loss:.4f}"
    )

Epoch [1/5] Loss: 0.6962
Epoch [2/5] Loss: 0.6941
Epoch [3/5] Loss: 0.6899
Epoch [4/5] Loss: 0.6787
Epoch [5/5] Loss: 0.6644


In [16]:
model.eval()

all_predictions = []
all_labels = []

with torch.no_grad():

    for xb, yb in test_loader:

        xb = xb.to(device)
        yb = yb.to(device)

        logits = model(xb)

        probabilities = torch.sigmoid(logits)

        predictions = (
            probabilities >= 0.5
        ).float()

        all_predictions.extend(
            predictions.cpu().numpy()
        )

        all_labels.extend(
            yb.cpu().numpy()
        )

In [17]:
accuracy = accuracy_score(
    all_labels,
    all_predictions
)

print(
    f"Test Accuracy: {accuracy * 100:.2f}%"
)

print(
    classification_report(
        all_labels,
        all_predictions,
        target_names=[
            "Negative",
            "Positive"
        ]
    )
)

Test Accuracy: 51.62%
              precision    recall  f1-score   support

    Negative       0.53      0.26      0.35      4940
    Positive       0.51      0.77      0.62      4977

    accuracy                           0.52      9917
   macro avg       0.52      0.52      0.48      9917
weighted avg       0.52      0.52      0.48      9917



In [18]:
torch.save(
    model.state_dict(),
    "sentiment_rnn.pt"
)

print("Model saved successfully")

Model saved successfully


In [19]:
def predict_sentiment(review):

    model.eval()

    # Same preprocessing used during training
    cleaned_review = clean_text(review)

    # Convert words → integers + padding
    encoded_review = encode_review(
        cleaned_review
    )

    # Convert to tensor
    input_tensor = torch.tensor(
        encoded_review,
        dtype=torch.long
    ).unsqueeze(0).to(device)

    with torch.no_grad():

        logits = model(input_tensor)

        probability = torch.sigmoid(
            logits
        ).item()

    if probability >= 0.5:

        sentiment = "Positive"
        confidence = probability

    else:

        sentiment = "Negative"
        confidence = 1 - probability

    print("Review:")
    print(review)

    print(
        "\nPredicted Sentiment:",
        sentiment
    )

    print(
        f"Confidence: {confidence * 100:.2f}%"
    )

In [24]:
review = """
This movie was absolutely amazing.
The story was engaging and the acting was excellent.
I really enjoyed watching it.
"""

predict_sentiment(review)

Review:

This movie was absolutely amazing.
The story was engaging and the acting was excellent.
I really enjoyed watching it.


Predicted Sentiment: Positive
Confidence: 58.20%
